# 7. Regression

Predict the exact NewsGuard score 0–100 (ridge; ModernBERT with squared error on score/100). Same split, same models for both tasks. Linear models: the penalty is chosen with 3 inner folds of the
training outlets. ModernBERT: 5 epochs, lr 3e-5, effective batch 32, last epoch kept. For every test article the
outputs are saved in `predictions/regression/<model>/`, and every fold model in `Modelli/20_FromScratch/`.

In [1]:
TASK = "regression"
MODELS = "TF-IDF,Qwen3-8B,Mistral-24B"     # "ModernBERT" runs on the GPU
N_JOBS = 16

In [2]:
# Parameters
MODELS = "ModernBERT"


In [3]:
import warnings

import numpy as np
import pandas as pd

import common as C

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
C.OUT.mkdir(parents=True, exist_ok=True)
LOG = C.OUT / "log.txt"
print(C.environment())

import models as MO

art, outlets = MO.load()

{'python': '3.12.3', 'seed': 42, 'split seed': 1, 'numpy': '2.5.4', 'pandas': '3.0.6', 'sklearn': '1.9.1', 'scipy': '1.18.1', 'torch': '2.5.1+cu121', 'transformers': '4.51.3', 'bitsandbytes': '0.50.2', 'trafilatura': '2.3.1'}


/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [4]:
for name in MODELS.split(","):
    for k in range(1, 6):
        if name == "ModernBERT":
            MO.bert_fold(TASK, art, k, "answerdotai/ModernBERT-base", name, LOG)
        else:
            MO.linear_fold(TASK, name, art, k, X=None if name == "TF-IDF" else MO.load_embeddings(name, art),
                           n_jobs=N_JOBS, logfile=LOG)

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


2026-11-11 23:33:02  regression, ModernBERT, fold 1, epoch 1/5: loss 0.0594 (1063 s)


2026-11-11 23:51:07  regression, ModernBERT, fold 1, epoch 2/5: loss 0.0198 (2148 s)


2026-11-12 00:09:59  regression, ModernBERT, fold 1, epoch 3/5: loss 0.0091 (3280 s)


2026-11-12 00:28:47  regression, ModernBERT, fold 1, epoch 4/5: loss 0.0040 (4408 s)


2026-11-12 00:47:25  regression, ModernBERT, fold 1, epoch 5/5: loss 0.0019 (5526 s)


... (25 outputs left out)

2026-11-12 05:58:01  regression, ModernBERT, fold 5, epoch 3/5: loss 0.0091 (2908 s)


2026-11-12 06:14:11  regression, ModernBERT, fold 5, epoch 4/5: loss 0.0050 (3878 s)


2026-11-12 06:30:26  regression, ModernBERT, fold 5, epoch 5/5: loss 0.0021 (4853 s)


2026-11-12 06:31:50  regression, ModernBERT, fold 5: done in 4937 s, GPU peak 10.0 GB


In [5]:
for name in MODELS.split(","):
    folder = C.PREDICTIONS / TASK / name
    if name == "ModernBERT":
        print(pd.concat([pd.read_csv(f).assign(fold=f.stem[-1]) for f in sorted(folder.glob("loss_fold*.csv"))])
              .pivot(index="epoch", columns="fold", values="loss").round(4))
    else:
        p = pd.concat([pd.read_csv(f) for f in sorted(folder.glob("penalty_fold*.csv"))])
        print(name, "chosen penalty per fold:", p[p.chosen].set_index("fold").penalty.to_dict())

fold        1       2       3       4       5
epoch                                        
1      0.0594  0.0581  0.0588  0.0675  0.0557
2      0.0198  0.0185  0.0218  0.0232  0.0197
3      0.0091  0.0081  0.0105  0.0119  0.0091
4      0.0040  0.0034  0.0046  0.0057  0.0050
5      0.0019  0.0017  0.0024  0.0027  0.0021
